<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/08-operations/02-reliability-and-fallbacks.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 可靠性与降级

**目标：** 用重试、退避、降级（fallback）和优雅降级（graceful degradation），让一个 LLM 功能在真实世界里活下来（速率限制、超时、瞬时错误、坏输出，以及服务商干脆挂了）。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

## 准备工作

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的 setup 辅助函数，并拉入 `groq` 客户端——这是本 notebook 唯一需要的依赖。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领免费 key（无需信用卡）；在 Colab 里通过**钥匙图标** → **Add new secret** 添加，名称严格写成 `GROQ_API_KEY`，并打开 notebook access。在本地则把 `GROQ_API_KEY` 设为环境变量。

（完整讲解见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 加载 GROQ_API_KEY 并返回一个就绪的 Groq 客户端。
client, MODEL = setup()

## 模型调用是你技术栈里最不可靠的东西

数据库查询很少失败，而且失败得很快。LLM API 调用正好相反：它慢、会被限流、偶尔发生瞬时错误、有时返回畸形输出，而且隔一阵子整个服务商都会有个糟糕的一小时。如果你的功能把这次调用当成「总是快速成功」来对待，它就会成为你最不可靠的组件。

LLM 的可靠性工程，大体上就是你面对任何不稳定远程依赖会用的那套工具（**重试、退避、超时、降级、熔断、优雅降级**），外加一个 LLM 特有的转折：一次调用可能「成功」（HTTP 200），却返回你的应用没法用的输出。我们会在 Groq 上把每一层搭起来。

## 第 1 层：指数退避 + 抖动的重试

速率限制（429）和瞬时 5xx 错误是*预期内*的，不是异常：正确的反应是等待并重试，而不是失败。用指数方式退避，免得往一个本就吃力的服务上猛砸；再加上**抖动（jitter）**，让一大批同时重试的客户端不会同步成惊群。只重试*可重试*的错误；其余的快速失败（一个 400 不会自己变好）。

In [ ]:
import time, random

# Groq 抛的是带类型的错误；我们重试其中的瞬时错误。（这里用无参 random 没问题——
# 这段是实时跑的，不在确定性工作流里。）
try:
    from groq import RateLimitError, APIStatusError, APIConnectionError
    RETRYABLE = (RateLimitError, APIConnectionError)
except Exception:                      # 让 notebook 对 SDK 版本漂移保持健壮
    RETRYABLE = ()

def call_with_retry(messages, max_attempts=4, base=0.5, **kwargs):
    for attempt in range(1, max_attempts + 1):
        try:
            return client.chat.completions.create(model=kwargs.pop("model", MODEL),
                                                   messages=messages, **kwargs)
        except RETRYABLE as e:
            if attempt == max_attempts:
                raise
            delay = base * (2 ** (attempt - 1)) + random.uniform(0, base)  # 退避 + 抖动
            print(f"  attempt {attempt} hit {type(e).__name__}; retrying in {delay:.2f}s")
            time.sleep(delay)

resp = call_with_retry([{"role": "user", "content": "Say 'reliable' and nothing else."}],
                       max_tokens=10)
print(resp.choices[0].message.content)

在健康的连接上，它在第 1 次尝试就返回。被限流时，你会看到重试那几行被触发，而调用最终仍然成功。这个模式（重试瞬时错误、退避、抖动、限制尝试次数、最后再抛出）是价值最高的单条可靠性习惯；把它包在每一次模型调用外面。

## 第 2 层：超时和一个降级模型

一次挂死的调用比一次失败更糟：它占着资源，还让用户干等。设一个**超时**，当主路径失败时（超时、重试耗尽，或模型不可用），**降级**到更小/更快的模型、更便宜的服务商，或一个缓存/预置的回答。一个打折扣的答案也好过一个转圈或一个 500。

In [ ]:
def answer_with_fallback(question, timeout=10):
    # 主模型：默认模型。降级：一个更小的。在真实应用里，降级目标
    # 可能是完全不同的服务商，这样一个厂商的故障不是你的故障。
    chain = [MODEL, "openai/gpt-oss-20b"]
    for i, model in enumerate(chain):
        try:
            resp = call_with_retry(
                [{"role": "user", "content": question}],
                model=model, max_tokens=120, timeout=timeout,
            )
            return {"answer": resp.choices[0].message.content, "served_by": model,
                    "degraded": i > 0}
        except Exception as e:
            print(f"  {model} failed ({type(e).__name__}); "
                  f"{'falling back' if i+1 < len(chain) else 'no fallback left'}")
    # 最后一道防线：绝不让用户什么都拿不到。
    return {"answer": "Sorry — I can't answer right now. Please try again shortly.",
            "served_by": None, "degraded": True}

print(answer_with_fallback("What is a subnet mask?"))

正常情况下由主模型来服务（`degraded: False`）。如果它失败了，你会透明地降到更小的模型；只有当*所有东西*都失败时，用户才会拿到那句诚实的预置消息，而绝不是一个未处理的异常。`degraded` 这个标志很重要：把它作为指标发出去（第 01 节），这样你就能*知道*你有多经常在降级状态下运行，哪怕用户没察觉。

## 第 3 层：验证输出，然后重试或降级（LLM 特有的那一层）

> **⚠️ 生产现实 ——** 普通服务没有的那个失败模式：**200 OK，内容却没法用。** 模型返回无效 JSON、一个不在名单上的枚举值，或者一个空答案。HTTP 调用「成功了」，但你的应用用不了这个结果。把它当作失败：验证，然后重试一次（下一次采样往往就好了），然后降级。

这是你在 `01-model-apis/01-structured-output` 里做的 schema 校验在可靠性上的对偶：同一个检查，现在接进了重试/降级机器里。

In [ ]:
import json, re

def get_structured(question, max_tries=3):
    prompt = (f'Answer as JSON: {{"answer": "<one sentence>", '
              f'"confidence": <0-1 number>}}. Question: {question}')
    for attempt in range(1, max_tries + 1):
        resp = call_with_retry([{"role": "user", "content": prompt}], max_tokens=120)
        raw = resp.choices[0].message.content
        try:
            obj = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
            assert isinstance(obj.get("answer"), str) and obj["answer"]
            assert isinstance(obj.get("confidence"), (int, float)) and 0 <= obj["confidence"] <= 1
            return obj                              # 有效 -> 完成
        except Exception:
            print(f"  attempt {attempt}: output failed validation, retrying")
    return {"answer": "unavailable", "confidence": 0.0, "degraded": True}  # 安全兜底

print(get_structured("What layer of the OSI model is TCP?"))

现在有两个不同的重试循环叠在一起：`call_with_retry` 处理*传输*失败（429/5xx），而这个循环处理*内容*失败（形状不对）。它们是不同的问题，但形状相同：重试、限制次数、然后降级到一个你的应用总能处理的安全兜底。

## 第 4 层：熔断器 —— 别再猛砸一个已经死掉的依赖

当服务商真的挂了，对每个请求都重试只是白白烧延迟和钱，还拖慢降级。**熔断器（circuit breaker）** 会注意到一连串失败，然后「打开」（在一个冷却窗口内直接快速失败到降级路径），接着「半开」来测试服务是否恢复了。这就是一次服务商故障让你每个请求付出*几秒*、还是付出你全部重试预算的区别。

In [ ]:
class CircuitBreaker:
    def __init__(self, threshold=5, cooldown_s=30):
        self.threshold, self.cooldown = threshold, cooldown_s
        self.fails, self.opened_at = 0, None
    def allow(self, now):
        if self.opened_at is None:
            return True                              # 关闭：正常
        if now - self.opened_at >= self.cooldown:
            self.opened_at = None; self.fails = 0    # 半开：给它一次机会
            return True
        return False                                 # 打开：快速失败
    def record(self, ok, now):
        if ok:
            self.fails = 0; self.opened_at = None
        else:
            self.fails += 1
            if self.fails >= self.threshold:
                self.opened_at = now                 # 跳闸

# 模拟：对「已死」的服务商发起 8 次调用。看熔断器跳闸并快速失败。
cb, clock = CircuitBreaker(threshold=3, cooldown_s=30), 0.0
for i in range(8):
    clock += 1
    if not cb.allow(clock):
        print(f"call {i}: circuit OPEN -> skip provider, go straight to fallback")
        continue
    ok = False                                       # 假装每次真实调用都失败
    cb.record(ok, clock)
    print(f"call {i}: tried provider, {'ok' if ok else 'FAILED'} (fails={cb.fails})")

三次调用失败后，熔断器打开，后续调用完全跳过那个死掉的服务商：立即降级，而不是每次都等完整个重试预算。冷却之后它半开，去探测是否恢复。在生产里，熔断器的开/闭状态也是一个**值得告警的指标**（第 01 节）：一个打开的熔断器意味着你正在降级运行。

## 可靠性清单

用这些把每一次生产模型调用包起来，大致按这个顺序：

1. **重试瞬时错误**（429/5xx/连接），用指数退避 + 抖动；对 4xx 快速失败。
2. **给每次调用设超时**；一次挂死比一次失败更糟。
3. **重试耗尽时降级**：更小的模型、不同服务商、缓存，或者一句诚实的预置回答。绝不把裸异常抛给用户。
4. **验证输出。** 一个 200 加没法用的内容是失败，所以要重试再降级。
5. **对持续失败的依赖做熔断**，让一次故障每个请求只付出几秒，而不是你整个重试预算。
6. **把 `degraded` 作为指标发出去**（第 01 节），好让你知道这些安全网有多经常在接住你。

除了第 4 层的输出验证，这些没有一样是 LLM 特有的。它就是经典的分布式系统卫生习惯，应用到你技术栈里最不稳定的那个依赖上——这也正是要点：一个把模型调用当成任何远程依赖、同样严格对待的 FDE，交付的东西能扛住与生产环境的接触。

## 练习

1. **让它真的去重试。** 制造一次速率限制（在免费档上用一串紧密的并行调用），确认 `call_with_retry` 能在裸调用会崩掉的地方恢复过来。记录它用了几次尝试。
2. **跨服务商的降级。** 改一下降级链，让第二项是*另一个服务商*的 SDK（哪怕只是打桩）。为什么同服务商内的降级无法保护你免受那个最重要的故障——服务商本身挂了？
3. **给重试算预算。** 重试会成倍放大成本和延迟。对一个 3s 的 p95 延迟 SLO、每次调用 p50 为 0.8s，在你必须放弃并降级之前，能负担几次重试？把它和 `08-operations/01` 里的延迟指标挂上。
4. **把所有东西组合起来。** 用熔断器和降级链把 `get_structured` 包起来，再用 notebook 01 里的 tracer 追踪它。现在你有了一个函数：重试传输、验证内容、优雅降级、故障时熔断，而且完全可观测——这就是一次真实生产 LLM 调用的形状。